# Class 9: Practicing Data Visualisation 

**In this notebook you will:**
1. Load real data from an experiment trying to predict CEFR levels with an LLM.
2. Look at a series of **intentionally bad** visualisations
3. Identify *what* makes each one poor
4. **Improve them** — applying principles of good scientific data visualisation

**Guide to the Basics:**
https://www.datacamp.com/tutorial/matplotlib-tutorial-python

**What is the data:**
The data comes from one of my recent experiments. We tried to predict CEFR levels using sentence embeddings of a modern large language model (Qwen3-embedding-8B).
We tried to fit some different models, linear regression, logistic regression, ordinal regression, neural network regression and neural network classifier.
These models were fitted on every fifth layer of the embeddings.
We tested the performance on different corpuses, to see if some corpuses/languages were harder to predict.
We are gonna try to make two basic plots from that data.
The data is currently very long -> 66200 lines long.
This is beacuse we have the original label for each essay and the prediction, for each combination of model, layer and corpus.
I have some functions that aggregate this into easier to understand formats for you: 'mae_by_group' & 'layer_mae'. 

'mae_by_group' is mean average accuracy by corpus, i.e., how wrong is the average prediction grouped by each corpus, for layer 15 and the linear regression model. A total of 10 numbers.

'layer_mae' finds the mean avereage accuracy for each combination of layer, model and corpus.

> **Columns**
> | Column | Description |
> |--------|-------------|
> | `model` | Statistical model used on the embeddings|
> | `layer` | Transformer layer the embedding was extracted from (1, 5, 10, ..., 30, 35) |
> | `dataset` | The CEFR dataset |
> | `y_true` | Gold CEFR level (0–4) (rated by examiners) |
> | `y_pred` | Model's continuous prediction (if a continous model) |
> | `y_pred_round` | Model's rounded prediction to match CEFR levels |


## 0 · Setup

In [2]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.ticker as ticker
import seaborn as sns
from sklearn.metrics import mean_absolute_error

# Load data
df = pd.read_csv("../data/CEFR_Prediction/IID_Predictions.csv")

print(df.shape)
df.head()

(66200, 6)


,model,layer,dataset,y_true,y_pred,y_pred_round
0,linear_regression,1,cefr-asag,2,2.231335,2
1,linear_regression,1,cefr-asag,3,2.051222,2
2,linear_regression,1,cefr-asag,2,1.641436,2
3,linear_regression,1,cefr-asag,1,0.769386,1
4,linear_regression,1,cefr-asag,2,1.387006,1


---
## Exercise 1 · Comparing model error across datasets
We want to know how well the model performs on each dataset.
Imagine that we are especially interested in how difficult the different datasets are to predict.
The data can vary in either, config (what kind of statistical model was used), layer (what layer of the LLM was used) and dataset.
Below is a first attempt using `linear_regression` at layer 15.


In [3]:
# Only pick the linear regression config for layer 15.
subset = df[(df["model"] == "linear_regression") & (df["layer"] == 15)]

# Get the mean absolute error for each OOD group.
mae_by_group = subset.groupby("dataset").apply(
    lambda g: mean_absolute_error(g["y_true"], g["y_pred"])
).reset_index(name="MAE")
print(mae_by_group.shape[0])
mae_by_group.head()

10


,dataset,MAE
0,cefr-asag,0.906630
1,cople2,0.475939
2,elle,0.519533
3,icle500,0.800722
4,merlin-all,0.511665


In [ ]:
# --- BAD PLOT ---
fig, ax = plt.subplots()
ax.bar(mae_by_group["dataset"], mae_by_group["MAE"])
ax.set_ylim(0.4, 0.92)
ax.set_title("MAE")
plt.show()

### 🔍 What's wrong?

Think about the following questions before reading on:

- Can you read the x-axis labels?
- Is the y-axis range appropriate?
- Does the chart title tell you anything useful?
- Are the datasets ordered in a meaningful way?
- Is a bar chart even the right choice here?

---

### ✏️ Your turn — improve this plot

**Goals:**
- Make all labels readable
- Add a meaningful, descriptive title and axis labels
- Order bars so the reader can immediately identify best/worst datasets
- Add error bars or jitter to show variance, not just the mean
- Consider colour: should all bars be the same colour?

In [ ]:
# Your improved version here
# Check out the documentation for matplotlib and seaborn for more ideas on how to customize the plot! https://matplotlib.org/stable/api/_as_gen/matplotlib.pyplot.barh.html
fig, ax = plt.subplots(figsize=(8, 5))
# TODO: order the bars with df.sort_values("column", ascending=False)

# TODO: Change 0.5 to the actual mean/median value and pick your own colors!
colors = ["blue" if m < 0.5 else "red" for m in mae_by_group["MAE"]]

# TODO: use ax.barh for horizontal bars, makes it easier to read the group names or rotate the x-axis labels with ax.set_xticklabels() and plt.xticks(rotation=45)
# documentation: https://matplotlib.org/stable/api/_as_gen/matplotlib.pyplot.barh.html

# TODO: add a vertical line for the mean/median MAE across groups using ax.axvline().

# Add labels and title (I normally add these in powerpoint after, but here we can do it in code)
ax.legend()
ax.set_xlabel("")
ax.set_ylabel("")
ax.set_title("")

# Show the plot
plt.tight_layout()
plt.show()

### 🛠️ REFERENCE SOLUTION — try on your own first!
Or pick elements to if you are stuck.

In [ ]:
# --- REFERENCE SOLUTION ---
fig, ax = plt.subplots(figsize=(8, 5))
# hint: order the bars with df.sort_values("column", ascending=False)
mae_by_group = mae_by_group.sort_values("MAE", ascending=False)
# hint: color bars differently based on whether they are above or below the mean/median
colors = ["#389f19c9" if m < mae_by_group["MAE"].mean() else "#d65f4d9c"
          for m in mae_by_group["MAE"]]
# hint: use ax.barh for horizontal bars, makes it easier to read the group names
ax.barh(mae_by_group["dataset"], mae_by_group["MAE"], color=colors)

# Add a vertical line for the mean
ax.axvline(mae_by_group["MAE"].mean(), color="gray", linestyle="--", label=f"Mean MAE = {mae_by_group['MAE'].mean():.2f}")

# Add labels and title (I normally add these in powerpoint after, but here we can do it in code)
ax.legend()
ax.set_xlabel("Mean Absolute Error")
ax.set_title("Probing Accuracy by Test Dataset\n"
             "Linear regression head · Layer 18 · Qwen3-Embedding-8B")

plt.tight_layout()
plt.show()

---
## Exercise 2 · How does performance change across transformer layers?
A key question in probing research: *which layer encodes linguistic information best?*  
Here is a first attempt at visualising MAE as a function of layer depth.


In [ ]:
# MAE grouped by each layer and model
layer_mae = df.groupby(["model", "layer"]).apply(
    lambda g: mean_absolute_error(g["y_true"], g["y_pred"])
).reset_index(name="MAE")
layer_mae.head()



# --- BAD PLOT ---
# One line of aggregated MAE values for each layer, averaged across all models.
fig, ax = plt.subplots()
sns.lineplot(data=layer_mae, x="layer", y="MAE", marker="o", color = "red", ax=ax)
plt.show()

### 🔍 What's wrong?

- What context is missing that a reader needs?
- Is the line enough, or should the uncertainty across models also be shown?
- How does the red colour choice affect readability?
- What annotation would help the reader find the "best" layer instantly?

---

### ✏️ Your turn — improve this plot

**Goals:**
- Add proper labels and a descriptive title
- Plot a **separate line per model** (to show variance across domains)
- Highlight the layer with the lowest overall MAE

In [ ]:
# Refrence solution - Do on your own first!
fig, ax = plt.subplots(figsize=(8, 5))

# Create a SNS.lineplot with markers for each layer's MAE for each model. (Hint: use the hue argument)
# Guide: https://www.datacamp.com/tutorial/python-seaborn-line-plot-tutorial




### 🛠️ REFERENCE SOLUTION — try on your own first!
Or pick elements to if you are stuck. This implementation is more complicated than needed. The entire part about the legend is easier to fix in PowerPoint.

In [ ]:
# Simple reference solution, feel free to customize the colors, markers, and styles!:
fig, ax = plt.subplots(figsize=(8, 5))

# Add color gradient sorted by final layer MAE (hint: use a dictionary to map model names to colors)
model_colors = {
    "linear_regression": "#E74C3C",
    "MLP_linear_regression": "#F07B1D",
    "ordinal_cumlink" : "#F5C518",
    "MLP_logistic_regression": "#A8D44A",
    "logistic_regression": "#2ECC71"
    }

# Add different markers for each model to make color blind friendly(hint: use the style argument)
model_markers = {
    "linear_regression":       "o",
    "MLP_linear_regression":   "s",
    "ordinal_cumlink":         "D",
    "MLP_logistic_regression": "^",
    "logistic_regression":     "P"
}

# Add order to the legend for better legend readability (hint: use the hue_order argument)
order = ["linear_regression", "MLP_linear_regression", "ordinal_cumlink", "MLP_logistic_regression", "logistic_regression"]
layer_mae["model"] = pd.Categorical(layer_mae["model"], categories=order, ordered=True)
layer_mae = layer_mae.sort_values("model")


# Add different markers for each model (hint: use the style argument)
sns.lineplot(
    data=layer_mae, x="layer", y="MAE",
    hue="model", style="model",          # style= drives marker/dash differentiation
    markers=model_markers,
    dashes=False,                         # solid lines only, no dashed styles
    palette=model_colors, ax=ax
)

# Fix the legend:
ax.legend(title="Model")

---
## Exercise 3: 
###  I would recommend spending some time on trying to create a visual 'experimental design' / 'flowchart'
### Otherwise, if that doesn't work with your exam, try to make a violin plot of predictions for each cefr_level.